# Big Data Storage and Spark SQL Analytics Workflow
**Curriculum:** 25MSDA3S01 – Big Data Analytics  
**Pillar:** Market & AI skill  
**Project objective:** Build a local Spark workflow, store an enterprise-style dataset in Hive-compatible tables, run Spark SQL analytics, visualize aggregated metrics, and measure query execution time.

> Dataset note: `data/retail_orders.csv` is a deterministic synthetic retail dataset created for this practical project. It is not real customer data.


## 1. Install and import dependencies

Run the install cell if PySpark is not already installed in your current Jupyter kernel. After installation, restart the kernel if the import fails.


In [ ]:
%pip install pyspark pandas matplotlib

In [ ]:
import os
import time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("PySpark imports successful")

## 2. Start a local Spark session with Hive support

`local[*]` uses the available local CPU cores. `enableHiveSupport()` enables Spark's Hive integration when the local Spark distribution has the required Hive support. A local embedded metastore may be created if no external metastore is configured.

For a shared enterprise Hive metastore, configure `hive-site.xml` and the metastore connection in Spark before creating the session.


In [ ]:
warehouse_dir = str(Path("spark_warehouse").resolve())
Path(warehouse_dir).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("BigDataStorageSparkSQLAnalytics")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Warehouse:", warehouse_dir)
print("Catalog databases:", [d.name for d in spark.catalog.listDatabases()])

## 3. Load and inspect the enterprise dataset

The data contains order details, city, product category, sales channel, quantity, price, discount, order status, delivery time and customer rating.


In [ ]:
data_path = Path("data/retail_orders.csv")
orders = spark.read.option("header", True).option("inferSchema", True).csv(str(data_path))

orders.printSchema()
print("Rows:", orders.count())
orders.show(5, truncate=False)

## 4. Data preparation and derived columns

Create date attributes and calculate gross and net sales in Spark. This keeps transformations distributed in Spark rather than collecting the full dataset into Pandas.


In [ ]:
orders = (
    orders
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("gross_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("net_amount", F.round(
        F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct")), 2
    ))
)

orders.select("order_id","order_date","city","product_category",
              "gross_amount","net_amount").show(5)

## 5. Store the dataset in a Hive-compatible table

The table is stored in the Spark catalog using the configured warehouse. When a Hive metastore is configured, Spark can use that metastore. In this standalone notebook, this is a local Hive-enabled Spark catalog, not a connection to a remote enterprise Hive cluster.


In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS retail_analytics")
spark.sql("USE retail_analytics")

(orders.write
    .mode("overwrite")
    .format("parquet")
    .saveAsTable("retail_orders"))

print("Tables in retail_analytics:")
spark.sql("SHOW TABLES").show(truncate=False)
print("Stored table row count:", spark.table("retail_orders").count())

## 6. Run analytical queries using Spark SQL

The queries below cover revenue by city, category performance, monthly sales, order status, and top products/categories. `EXPLAIN FORMATTED` can be used to inspect the physical execution plan.


In [ ]:
# Query 1: City-wise sales performance
city_revenue = spark.sql("""
    SELECT city,
           COUNT(*) AS total_orders,
           ROUND(SUM(net_amount), 2) AS total_revenue,
           ROUND(AVG(net_amount), 2) AS average_order_value
    FROM retail_analytics.retail_orders
    GROUP BY city
    ORDER BY total_revenue DESC
""")
city_revenue.show()

In [ ]:
# Query 2: Product category performance
category_performance = spark.sql("""
    SELECT product_category,
           COUNT(*) AS total_orders,
           ROUND(SUM(net_amount), 2) AS total_revenue,
           ROUND(AVG(delivery_days), 2) AS average_delivery_days,
           ROUND(AVG(customer_rating), 2) AS average_rating
    FROM retail_analytics.retail_orders
    GROUP BY product_category
    ORDER BY total_revenue DESC
""")
category_performance.show()

In [ ]:
# Query 3: Monthly revenue
monthly_revenue = spark.sql("""
    SELECT order_year, order_month,
           COUNT(*) AS total_orders,
           ROUND(SUM(net_amount), 2) AS monthly_revenue
    FROM retail_analytics.retail_orders
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
""")
monthly_revenue.show(24)

In [ ]:
# Query 4: Order status summary
status_summary = spark.sql("""
    SELECT order_status,
           COUNT(*) AS order_count,
           ROUND(SUM(net_amount), 2) AS total_revenue,
           ROUND(AVG(customer_rating), 2) AS average_rating
    FROM retail_analytics.retail_orders
    GROUP BY order_status
    ORDER BY order_count DESC
""")
status_summary.show()

In [ ]:
# Query 5: Top 3 cities by revenue within each product category
top_cities = spark.sql("""
    SELECT product_category, city, total_revenue, revenue_rank
    FROM (
        SELECT product_category, city,
               SUM(net_amount) AS total_revenue,
               DENSE_RANK() OVER (
                   PARTITION BY product_category
                   ORDER BY SUM(net_amount) DESC
               ) AS revenue_rank
        FROM retail_analytics.retail_orders
        GROUP BY product_category, city
    ) ranked
    WHERE revenue_rank <= 3
    ORDER BY product_category, revenue_rank
""")
top_cities.show(30, truncate=False)

## 7. Visualize aggregated metrics

Only the aggregated query results are collected to Pandas for plotting. The full dataset remains in Spark.


In [ ]:
city_pd = city_revenue.toPandas()
plt.figure(figsize=(9,5))
plt.bar(city_pd["city"], city_pd["total_revenue"])
plt.title("Total Net Revenue by City")
plt.xlabel("City")
plt.ylabel("Net revenue")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
monthly_pd = monthly_revenue.toPandas()
monthly_pd["period"] = (monthly_pd["order_year"].astype(str) + "-" +
                        monthly_pd["order_month"].astype(str).str.zfill(2))
plt.figure(figsize=(10,5))
plt.plot(monthly_pd["period"], monthly_pd["monthly_revenue"], marker="o")
plt.title("Monthly Net Revenue")
plt.xlabel("Month")
plt.ylabel("Net revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
category_pd = category_performance.toPandas()
plt.figure(figsize=(9,5))
plt.bar(category_pd["product_category"], category_pd["total_revenue"])
plt.title("Revenue by Product Category")
plt.xlabel("Product category")
plt.ylabel("Net revenue")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 8. Measure query execution time

This section measures wall-clock time for each query action. Spark is lazy, so the action (`count`) is included in the timing. The first query may include startup or warm-up overhead. Repeat measurements and compare like-for-like workloads before drawing performance conclusions.


In [ ]:
performance_queries = {
    "City aggregation": """
        SELECT city, SUM(net_amount) AS revenue
        FROM retail_analytics.retail_orders GROUP BY city
    """,
    "Category aggregation": """
        SELECT product_category, AVG(customer_rating) AS rating
        FROM retail_analytics.retail_orders GROUP BY product_category
    """,
    "Monthly aggregation": """
        SELECT order_year, order_month, SUM(net_amount) AS revenue
        FROM retail_analytics.retail_orders
        GROUP BY order_year, order_month
    """
}

timings = []
for query_name, sql_text in performance_queries.items():
    start = time.perf_counter()
    result_count = spark.sql(sql_text).count()  # action triggers execution
    elapsed = time.perf_counter() - start
    timings.append({
        "query": query_name,
        "execution_time_seconds": elapsed,
        "result_rows": result_count
    })

performance_df = pd.DataFrame(timings)
display(performance_df)

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(performance_df["query"], performance_df["execution_time_seconds"])
plt.title("Spark SQL Query Execution Time (Local Run)")
plt.xlabel("Query")
plt.ylabel("Execution time (seconds)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 9. Inspect a Spark SQL execution plan

The execution plan shows how Spark plans the query. The physical plan can include scans, exchanges/shuffles, hash aggregates, and sorts depending on the query and Spark version.


In [ ]:
spark.sql("""
    SELECT city, SUM(net_amount) AS revenue
    FROM retail_analytics.retail_orders
    GROUP BY city
""").explain("formatted")

## 10. Save analytical outputs

Export aggregate results, execution timings, and the Spark table data for reproducibility. The notebook's charts are generated when the cells are run.


In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

city_revenue.toPandas().to_csv(output_dir / "city_revenue_summary.csv", index=False)
category_performance.toPandas().to_csv(output_dir / "category_performance.csv", index=False)
status_summary.toPandas().to_csv(output_dir / "order_status_summary.csv", index=False)
monthly_revenue.toPandas().to_csv(output_dir / "monthly_revenue.csv", index=False)
performance_df.to_csv(output_dir / "query_execution_timings.csv", index=False)

print("Saved analytical outputs to:", output_dir.resolve())

## 11. Conclusion

This project demonstrates a local big-data workflow using Apache Spark and Spark SQL. A retail orders dataset is loaded, transformed, stored as a Hive-compatible catalog table, queried using SQL aggregations and window functions, and visualized using charts. Query execution times and a physical execution plan provide a basic view of Spark query performance.

**Limitations and deployment note:** The included setup uses local Spark (`local[*]`) and a local warehouse/metastore. It demonstrates Hive-enabled table storage through Spark's catalog. A production or shared Hive deployment requires a configured Hive Metastore service, compatible Hive configuration (`hive-site.xml`), and suitable permissions. The local execution-time chart is a measurement from the machine running the notebook, not a benchmark against a distributed cluster.
